# Fine-tune YOLOv8n-cls on CIFAR10

This notebook prepares CIFAR10 in Ultralytics classification folder format and fine-tunes `yolov8n-cls.pt` so it can be used as a task-faithful teacher model.

In [1]:
from pathlib import Path
import shutil

import torch
from ultralytics import YOLO

seed = 42
torch.manual_seed(seed)

work_dir = Path("./data/cifar10_yolov8n_cls")
model_checkpoint = "yolov8n-cls.pt"

epochs = 20
batch_size = 64
imgsz = 32
workers = 4
run_name = "yolov8n_cls_cifar10_finetune"

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")
print(f"Run workspace: {work_dir.resolve()}")

Using device: cuda
Run workspace: /workspace/mase-kd/cw/data/cifar10_yolov8n_cls


In [ ]:
model = YOLO(model_checkpoint)

train_results = model.train(
    data="cifar10",
    epochs=epochs,
    imgsz=imgsz,
    batch=batch_size,
    workers=workers,
    device=device,
    project=str(work_dir / "runs"),
    name=run_name,
)

print("Training complete.")
print("Run directory:", model.trainer.save_dir)
print("Best checkpoint:", model.trainer.best)
print("Last checkpoint:", model.trainer.last)

New https://pypi.org/project/ultralytics/8.4.19 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.235 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=cifar10, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=32, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_cls_cifar10_finetune4, nbs=64, nms=F

In [ ]:
best_ckpt = Path(model.trainer.best)
finetuned_model = YOLO(str(best_ckpt)) if best_ckpt.exists() else model

val_results = finetuned_model.val(
    data="cifar10",
    split="val",
    imgsz=imgsz,
    batch=batch_size,
    device=device,
)

print("Validation complete.")
print(val_results)

In [ ]:
teacher_dir = work_dir / "checkpoints"
teacher_dir.mkdir(parents=True, exist_ok=True)
teacher_export_path = teacher_dir / "yolov8n-cls-cifar10-teacher.pt"

if best_ckpt.exists():
    shutil.copy2(best_ckpt, teacher_export_path)
    teacher_checkpoint = str(teacher_export_path.resolve())
    print("Saved fine-tuned teacher checkpoint:")
    print(teacher_checkpoint)
else:
    teacher_checkpoint = model_checkpoint
    print("Best checkpoint not found after training; using base checkpoint:")
    print(teacher_checkpoint)